# ECA-DNet: Efficient Channel Attention Dehazing Network
## Testing— 
**Datasets Required**
1. `kmljts/reside-6k`
2. `balraj98/indoor-training-set-its-residestandard`
3. `balraj98/synthetic-objective-testing-set-sots-reside`


In [ ]:
import os, sys, cv2, time, json, random, warnings, gc, csv, argparse
import numpy as np
warnings.filterwarnings('ignore')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 0 — ARGUMENT PARSING & CONFIG (KAGGLE SAFE)
# ═══════════════════════════════════════════════════════════════
import argparse
import random
import numpy as np

def parse_args():
    p = argparse.ArgumentParser(description='ECA-LDNet Testing Suite')

    p.add_argument('--model', type=str, default=None,
                   help='Path to .keras model file')

    p.add_argument('--reside6k', type=str, default=None,
                   help='Path to RESIDE-6K root')

    p.add_argument('--its', type=str, default=None,
                   help='Path to ITS root')

    p.add_argument('--sots', type=str, default=None,
                   help='Path to SOTS root')

    p.add_argument('--output', type=str, default='./test_results',
                   help='Output directory for results')

    p.add_argument('--limit', type=int, default=None,
                   help='Limit number of test images per dataset')

    p.add_argument('--no-tta', action='store_true',
                   help='Disable test-time augmentation')

    p.add_argument('--history-csv', type=str, default=None,
                   help='Path to combined_training_history.csv')

    # ✅ FIX: Kaggle/Jupyter safe parsing
    args, _ = p.parse_known_args()
    return args


IMG_SIZE = 256
CHANNELS = 3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ═══════════════════════════════════════════════════════════════
# SECTION 1 — TENSORFLOW SETUP & CUSTOM LAYERS
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("ECA-LDNet — Comprehensive Testing Suite")
print("=" * 70)

import tensorflow as tf
from tensorflow.keras import layers, Model

# GPU setup (Kaggle safe)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {len(gpus)}")
if gpus:
    print(f"GPU name   : {gpus[0].name}")


# --- Custom Layers (must match training exactly) ---

class ECABlock(layers.Layer):
    """Efficient Channel Attention — learns WHICH channels matter."""

    def __init__(self, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.conv = layers.Conv1D(
            1,
            self.kernel_size,
            padding='same',
            use_bias=False
        )
        self.gap = layers.GlobalAveragePooling2D()

    def call(self, x):
        B = tf.shape(x)[0]

        gap = self.gap(x)
        gap = tf.reshape(gap, [B, self.channels, 1])

        attn = tf.sigmoid(self.conv(gap))
        attn = tf.reshape(attn, [B, 1, 1, self.channels])

        return x * attn

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'kernel_size': self.kernel_size})
        return cfg


class PixelAttention(layers.Layer):
    """Spatial Attention — learns WHERE to focus."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        C = input_shape[-1]
        self.conv1 = layers.Conv2D(max(8, C // 4), 1, padding='same', use_bias=False)
        self.conv2 = layers.Conv2D(1, 1, padding='same', activation='sigmoid')

    def call(self, x):
        attn = tf.nn.relu(self.conv1(x))
        attn = self.conv2(attn)
        return x * attn

    def get_config(self):
        return super().get_config()


class PhysicsCorrectionLayer(layers.Layer):
    """Atmospheric scattering model correction: J(x) = (I(x) - A) / t(x) + A"""

    def __init__(self, eps=0.1, blend=0.08, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        self.blend = blend

    def call(self, inputs):
        img, A, t = inputs

        A_bc = tf.reshape(A, [-1, 1, 1, 3])
        A_bc = tf.broadcast_to(A_bc, tf.shape(img))

        t_bc = tf.broadcast_to(t, tf.shape(img))

        j = (img - A_bc) / (t_bc + self.eps) + A_bc
        j = tf.clip_by_value(j, 0.0, 1.0)

        return img * (1 - self.blend) + j * self.blend

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'eps': self.eps, 'blend': self.blend})
        return cfg


CUSTOM_OBJECTS = {
    'ECABlock': ECABlock,
    'PixelAttention': PixelAttention,
    'PhysicsCorrectionLayer': PhysicsCorrectionLayer,
}

In [ ]:
import keras
import tensorflow as tf

print("Keras:", keras.__version__)
print("TensorFlow:", tf.__version__)

In [ ]:

# ═══════════════════════════════════════════════════════════════
# SECTION 2 — DATASET PATH DETECTION
# ═══════════════════════════════════════════════════════════════
def find_dir(base, candidates):
    """Auto-detect dataset directories (handles nested Kaggle structures)."""
    if not base or not os.path.isdir(base):
        return None
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p) and len(os.listdir(p)) > 0:
            return p
    for sub in os.listdir(base):
        sp = os.path.join(base, sub)
        if os.path.isdir(sp):
            for c in candidates:
                p = os.path.join(sp, c)
                if os.path.isdir(p) and len(os.listdir(p)) > 0:
                    return p
    return None

def setup_dataset_paths(args):
    """Configure dataset paths for Kaggle or local."""
    paths = {}

    # RESIDE-6K
    r6k_base = args.reside6k or '/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K'
    paths['R6K_TEST_HAZY'] = find_dir(r6k_base, ['test/hazy', 'Test/hazy'])
    paths['R6K_TEST_GT'] = find_dir(r6k_base, ['test/GT', 'test/gt', 'test/clear'])

    # ITS (for reference — usually training set, but some have test splits)
    its_base = args.its or '/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard'
    paths['ITS_HAZY'] = find_dir(its_base, ['hazy', 'Hazy'])
    paths['ITS_GT'] = find_dir(its_base, ['clear', 'Clear', 'GT', 'gt'])

    # SOTS
    sots_base = args.sots or '/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside'
    paths['SOTS_IN_HAZY'] = find_dir(sots_base, ['indoor/hazy', 'SOTS/indoor/hazy'])
    paths['SOTS_IN_GT'] = find_dir(sots_base, ['indoor/clear', 'indoor/gt', 'indoor/GT'])
    paths['SOTS_OUT_HAZY'] = find_dir(sots_base, ['outdoor/hazy', 'SOTS/outdoor/hazy'])
    paths['SOTS_OUT_GT'] = find_dir(sots_base, ['outdoor/clear', 'outdoor/Clear', 'outdoor/GT'])

    print("\n" + "=" * 65)
    print("DATASET PATH VERIFICATION")
    print("=" * 65)
    for name, path in paths.items():
        if path:
            n = len(os.listdir(path))
            print(f"  OK  {name:<18} {path}  [{n} files]")
        else:
            print(f"  XX  {name:<18} NOT FOUND")
    print("=" * 65)
    return paths

# ═══════════════════════════════════════════════════════════════
# SECTION 3 — IMAGE LOADING & PREPROCESSING
# ═══════════════════════════════════════════════════════════════
def preprocess_hazy(img):
    """Exact same preprocessing as training — gamma + per-channel normalization."""
    img = np.power(np.clip(img, 0, 1), 0.9).astype(np.float32)
    for c in range(3):
        lo, hi = img[:, :, c].min(), img[:, :, c].max()
        if hi > lo + 1e-6:
            img[:, :, c] = (img[:, :, c] - lo) / (hi - lo)
    return np.clip(img, 0, 1).astype(np.float32)

def load_image_pair(hazy_path, gt_path, size=IMG_SIZE):
    """Load and preprocess a hazy-GT image pair."""
    hazy = cv2.imread(str(hazy_path))
    gt = cv2.imread(str(gt_path))
    if hazy is None or gt is None:
        return None, None
    hazy = cv2.cvtColor(hazy, cv2.COLOR_BGR2RGB)
    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
    hazy = cv2.resize(hazy, (size, size)).astype(np.float32) / 255.0
    gt = cv2.resize(gt, (size, size)).astype(np.float32) / 255.0
    hazy = preprocess_hazy(hazy)
    return hazy, gt

def get_gt_path(hazy_name, gt_dir):
    """Find corresponding ground truth file for a hazy image."""
    from pathlib import Path
    stem = Path(hazy_name).stem
    candidates = [stem, stem.split('_')[0], stem.replace('_hazy', ''),
                  stem.replace('_fog', ''), stem.replace('_synthetic', '')]
    for cand in candidates:
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.bmp']:
            p = os.path.join(gt_dir, cand + ext)
            if os.path.exists(p):
                return p
    return None

In [ ]:

# ═══════════════════════════════════════════════════════════════
# SECTION 4 — INFERENCE UTILITIES
# ═══════════════════════════════════════════════════════════════
def gaussian_window_2d(size, sigma=0.35):
    """Gaussian blending window for sliding-window inference."""
    c = np.linspace(-1, 1, size)
    g = np.exp(-(c ** 2) / (2 * sigma ** 2))
    w = np.outer(g, g)
    return (w / w.max()).astype(np.float32)

def sliding_window_predict(model, img, patch=IMG_SIZE, stride=64, batch_size=8):
    """Process full-resolution images with overlapping patches."""
    H, W = img.shape[:2]
    output = np.zeros((H, W, 3), np.float32)
    weight = np.zeros((H, W, 3), np.float32)
    win3d = gaussian_window_2d(patch)[:, :, np.newaxis]
    ys = list(range(0, max(H - patch, 1), stride))
    xs = list(range(0, max(W - patch, 1), stride))
    if not ys or ys[-1] != max(H - patch, 0):
        ys.append(max(H - patch, 0))
    if not xs or xs[-1] != max(W - patch, 0):
        xs.append(max(W - patch, 0))
    coords = [(y, x) for y in ys for x in xs]
    crops = np.stack([img[y:y + patch, x:x + patch] for y, x in coords]).astype(np.float32)
    preds = model.predict(crops, batch_size=batch_size, verbose=0)
    for pred_p, (y, x) in zip(preds, coords):
        output[y:y + patch, x:x + patch] += pred_p * win3d
        weight[y:y + patch, x:x + patch] += win3d
    return np.clip(output / (weight + 1e-8), 0, 1)

def predict_full(model, img, patch=IMG_SIZE, stride=64, batch_size=8, tta=True):
    """Full prediction with optional Test-Time Augmentation (4-way)."""
    H, W = img.shape[:2]

    def _pred(im):
        if H <= patch and W <= patch:
            return model.predict(im[np.newaxis], verbose=0)[0].astype(np.float32)
        return sliding_window_predict(model, im, patch, stride, batch_size)

    out = _pred(img)
    if tta:
        o_lr = _pred(img[:, ::-1, :])[:, ::-1, :]
        o_ud = _pred(img[::-1, :, :])[::-1, :, :]
        o_r = np.rot90(_pred(np.rot90(img, 1)), -1)
        out = (out + o_lr + o_ud + o_r) / 4.0
    return np.clip(out, 0, 1).astype(np.float32)

def predict_single(model, img):
    """Simple single-pass prediction (no TTA, no sliding window)."""
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
    img_resized = preprocess_hazy(img_resized)
    pred = model.predict(img_resized[np.newaxis], verbose=0)[0]
    return np.clip(pred, 0, 1).astype(np.float32)

# ═══════════════════════════════════════════════════════════════
# SECTION 5 — METRICS
# ═══════════════════════════════════════════════════════════════
def compute_psnr(pred, gt):
    return float(
        tf.image.psnr(
            pred[np.newaxis],
            gt[np.newaxis],
            max_val=1.0
        ).numpy()[0]
    )
def compute_ssim(pred, gt):
    return tf.image.ssim(
        pred[np.newaxis],
        gt[np.newaxis],
        max_val=1.0
    ).numpy().item()
def compute_mae(pred, gt):
    return float(np.mean(np.abs(pred - gt)))

def compute_ms_ssim(pred, gt):
    """Multi-Scale SSIM (more robust quality measure)."""
    try:
        val = tf.image.ssim_multiscale(
                pred[np.newaxis],
                gt[np.newaxis],
                max_val=1.0
            ).numpy().item()
        return val if not np.isnan(val) else compute_ssim(pred, gt)
    except:
        return compute_ssim(pred, gt)

def compute_per_channel_psnr(pred, gt):
    """PSNR for each R, G, B channel separately."""
    results = {}
    for i, ch in enumerate(['R', 'G', 'B']):
        p = pred[:, :, i:i + 1]
        g = gt[:, :, i:i + 1]
        results[f'PSNR_{ch}'] = tf.image.psnr(
                                p[np.newaxis],
                                g[np.newaxis],
                                max_val=1.0
                            ).numpy().item()
    return results

def compute_sharpness(img):
    """Laplacian variance — higher = sharper."""
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

# ═══════════════════════════════════════════════════════════════
# SECTION 6 — FLOPs & LATENCY ANALYSIS
# ═══════════════════════════════════════════════════════════════
def compute_model_flops(model):
    """Estimate FLOPs using TF profiler."""
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
        forward_pass = tf.function(model.call, input_signature=[
            tf.TensorSpec(shape=(1, IMG_SIZE, IMG_SIZE, 3))])
        graph_info = profile(forward_pass.get_concrete_function().graph,
                             options=ProfileOptionBuilder.float_operation())
        flops = graph_info.total_float_ops
        return flops
    except Exception as e:
        print(f"  FLOPs estimation failed: {e}")
        # Fallback: rough estimate based on params
        params = model.count_params()
        est_flops = params * IMG_SIZE * IMG_SIZE * 2  # very rough
        return None

def measure_latency(model, n_runs=50, warmup=10):
    """Measure inference latency (ms per image)."""
    dummy = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    # Warmup
    for _ in range(warmup):
        model.predict(dummy, verbose=0)
    # Timed runs
    timings = []
    for _ in range(n_runs):
        t0 = time.time()
        model.predict(dummy, verbose=0)
        timings.append((time.time() - t0) * 1000)
    avg = np.mean(timings)
    std = np.std(timings)
    fps = 1000.0 / avg
    return {'avg_ms': avg, 'std_ms': std, 'fps': fps, 'timings': timings}

# ═══════════════════════════════════════════════════════════════
# SECTION 7 — DATASET EVALUATION
# ═══════════════════════════════════════════════════════════════
def evaluate_dataset(model, hazy_dir, gt_dir, name, limit=None, tta=True, save_csv=None):
    """Evaluate model on a full dataset with all metrics."""
    if not hazy_dir or not gt_dir:
        print(f"  SKIP {name} — dataset not found")
        return None
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    if limit:
        files = files[:limit]
    print(f"\n{'─' * 60}")
    print(f"Evaluating [{name}] — {len(files)} images, TTA={tta}")
    print(f"{'─' * 60}")

    records = []
    t_start = time.time()
    for i, fname in enumerate(files):
        hp = os.path.join(hazy_dir, fname)
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_image_pair(hp, gp)
        if hazy is None:
            continue

        pred = predict_full(model, hazy, tta=tta)

        # Core metrics
        psnr = compute_psnr(pred, gt)
        ssim = compute_ssim(pred, gt)
        mae = compute_mae(pred, gt)
        ms_ssim = compute_ms_ssim(pred, gt)

        # Per-channel
        ch_psnr = compute_per_channel_psnr(pred, gt)

        # Sharpness
        sharp_hazy = compute_sharpness(hazy)
        sharp_pred = compute_sharpness(pred)
        sharp_gt = compute_sharpness(gt)

        row = {
            'file': fname, 'PSNR': psnr, 'SSIM': ssim, 'MAE': mae,
            'MS_SSIM': ms_ssim,
            'PSNR_R': ch_psnr['PSNR_R'], 'PSNR_G': ch_psnr['PSNR_G'],
            'PSNR_B': ch_psnr['PSNR_B'],
            'Sharpness_Hazy': sharp_hazy, 'Sharpness_Pred': sharp_pred,
            'Sharpness_GT': sharp_gt,
        }
        records.append(row)

        if (i + 1) % 50 == 0 or (i + 1) == len(files):
            avg_p = np.mean([r['PSNR'] for r in records])
            avg_s = np.mean([r['SSIM'] for r in records])
            elapsed = time.time() - t_start
            print(f"  [{i + 1:>4}/{len(files)}] PSNR={avg_p:.4f} SSIM={avg_s:.4f} "
                  f"({elapsed:.1f}s)")

    if not records:
        print(f"  No valid pairs found for {name}")
        return None

    # Summary
    psnrs = [r['PSNR'] for r in records]
    ssims = [r['SSIM'] for r in records]
    maes = [r['MAE'] for r in records]
    ms_ssims = [r['MS_SSIM'] for r in records]

    print(f"\n  ╔═══ {name} RESULTS ═══")
    print(f"  ║ Images    : {len(records)}")
    print(f"  ║ PSNR      : {np.mean(psnrs):.4f} ± {np.std(psnrs):.4f} dB")
    print(f"  ║ SSIM      : {np.mean(ssims):.4f} ± {np.std(ssims):.4f}")
    print(f"  ║ MAE       : {np.mean(maes):.6f} ± {np.std(maes):.6f}")
    print(f"  ║ MS-SSIM   : {np.mean(ms_ssims):.4f} ± {np.std(ms_ssims):.4f}")
    print(f"  ║ PSNR [R]  : {np.mean([r['PSNR_R'] for r in records]):.4f}")
    print(f"  ║ PSNR [G]  : {np.mean([r['PSNR_G'] for r in records]):.4f}")
    print(f"  ║ PSNR [B]  : {np.mean([r['PSNR_B'] for r in records]):.4f}")
    print(f"  ║ Sharpness : Hazy={np.mean([r['Sharpness_Hazy'] for r in records]):.1f}"
          f"  Pred={np.mean([r['Sharpness_Pred'] for r in records]):.1f}"
          f"  GT={np.mean([r['Sharpness_GT'] for r in records]):.1f}")
    print(f"  ╚{'═' * 50}")

    # Save CSV
    if save_csv:
        with open(save_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=records[0].keys())
            writer.writeheader()
            writer.writerows(records)
        print(f"  Saved: {save_csv}")

    return records


In [ ]:

# ═══════════════════════════════════════════════════════════════
# SECTION 8 — TTA vs NO-TTA COMPARISON
# ═══════════════════════════════════════════════════════════════
def compare_tta(model, hazy_dir, gt_dir, name, n_samples=50, save_csv=None):
    """Compare TTA vs no-TTA performance."""
    if not hazy_dir or not gt_dir:
        return None
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])[:n_samples]
    print(f"\n{'─' * 60}")
    print(f"TTA Comparison [{name}] — {len(files)} images")
    print(f"{'─' * 60}")

    records = []
    for fname in files:
        hp = os.path.join(hazy_dir, fname)
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_image_pair(hp, gp)
        if hazy is None:
            continue

        pred_no_tta = predict_full(model, hazy, tta=False)
        pred_tta = predict_full(model, hazy, tta=True)

        records.append({
            'file': fname,
            'PSNR_no_TTA': compute_psnr(pred_no_tta, gt),
            'SSIM_no_TTA': compute_ssim(pred_no_tta, gt),
            'PSNR_TTA': compute_psnr(pred_tta, gt),
            'SSIM_TTA': compute_ssim(pred_tta, gt),
        })

    if records:
        avg_no = np.mean([r['PSNR_no_TTA'] for r in records])
        avg_tta = np.mean([r['PSNR_TTA'] for r in records])
        print(f"  No TTA : PSNR={avg_no:.4f}  SSIM={np.mean([r['SSIM_no_TTA'] for r in records]):.4f}")
        print(f"  TTA    : PSNR={avg_tta:.4f}  SSIM={np.mean([r['SSIM_TTA'] for r in records]):.4f}")
        print(f"  Gain   : +{avg_tta - avg_no:.4f} dB PSNR")
        if save_csv:
            with open(save_csv, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=records[0].keys())
                writer.writeheader()
                writer.writerows(records)
    return records

# ═══════════════════════════════════════════════════════════════
# SECTION 9 — VISUAL COMPARISON PLOTS
# ═══════════════════════════════════════════════════════════════
def plot_visual_comparison(model, hazy_dir, gt_dir, save_path, title, tta=True, n=6):
    """Generate best/worst visual comparison grid."""
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        print("  matplotlib not available, skipping visual plots")
        return

    if not hazy_dir or not gt_dir:
        return
    files = sorted(os.listdir(hazy_dir))
    random.shuffle(files)
    samples = []
    for fname in files[:200]:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_image_pair(os.path.join(hazy_dir, fname), gp)
        if hazy is None:
            continue
        pred = predict_full(model, hazy, tta=tta)
        pv = compute_psnr(pred, gt)
        sv = compute_ssim(pred, gt)
        samples.append((pv, sv, hazy, pred, gt, fname))
        if len(samples) >= 30:
            break

    if len(samples) < 6:
        print(f"  Not enough samples for {title}")
        return

    samples.sort(key=lambda x: x[0])
    show = samples[:3] + samples[-3:]
    labels = ['Worst'] * 3 + ['Best'] * 3

    fig, axes = plt.subplots(6, 4, figsize=(20, 30))
    for row, (pv, sv, hazy, pred, gt, fname) in enumerate(show):
        error = np.abs(pred - gt)
        axes[row, 0].imshow(np.clip(hazy, 0, 1))
        axes[row, 0].set_title(f'[{labels[row]}] Hazy\n{fname}', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(np.clip(pred, 0, 1))
        axes[row, 1].set_title(f'Predicted\nPSNR={pv:.2f} SSIM={sv:.4f}', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(np.clip(gt, 0, 1))
        axes[row, 2].set_title('Ground Truth', fontsize=9)
        axes[row, 2].axis('off')

        im = axes[row, 3].imshow(error, cmap='hot', vmin=0, vmax=0.3)
        axes[row, 3].set_title(f'Error Map\nMAE={np.mean(error):.4f}', fontsize=9)
        axes[row, 3].axis('off')
        plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04)

    plt.suptitle(f'{title}\n3 Worst (top) vs 3 Best (bottom)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Visual comparison saved: {save_path}")

In [ ]:

# ═══════════════════════════════════════════════════════════════
# SECTION 10 — ERROR HEATMAP ANALYSIS
# ═══════════════════════════════════════════════════════════════
def plot_error_heatmaps(model, hazy_dir, gt_dir, save_path, n_samples=50, tta=True):
    """Average error heatmap — shows WHERE the model struggles spatially."""
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        return

    if not hazy_dir or not gt_dir:
        return
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])[:n_samples]

    error_sum = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float64)
    count = 0
    for fname in files:
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_image_pair(os.path.join(hazy_dir, fname), gp)
        if hazy is None:
            continue
        pred = predict_full(model, hazy, tta=tta)
        error_sum += np.abs(pred.astype(np.float64) - gt.astype(np.float64))
        count += 1

    if count == 0:
        return
    avg_error = (error_sum / count).astype(np.float32)

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    # Overall
    im0 = axes[0].imshow(np.mean(avg_error, axis=2), cmap='hot', vmin=0, vmax=0.05)
    axes[0].set_title(f'Average Error (All Channels)\nn={count} images', fontsize=12)
    plt.colorbar(im0, ax=axes[0])
    # Per channel
    for i, (ch, cmap) in enumerate(zip(['Red', 'Green', 'Blue'], ['Reds', 'Greens', 'Blues'])):
        im = axes[i + 1].imshow(avg_error[:, :, i], cmap=cmap, vmin=0, vmax=0.05)
        axes[i + 1].set_title(f'{ch} Channel Error', fontsize=12)
        plt.colorbar(im, ax=axes[i + 1])
    for ax in axes:
        ax.axis('off')
    plt.suptitle('Spatial Error Distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Error heatmaps saved: {save_path}")

# ═══════════════════════════════════════════════════════════════
# SECTION 11 — PIXEL DISTRIBUTION & SHARPNESS
# ═══════════════════════════════════════════════════════════════
def plot_pixel_distribution(model, hazy_dir, gt_dir, save_path, n_samples=50, tta=False):
    """Pixel intensity histogram + sharpness comparison."""
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        return

    if not hazy_dir or not gt_dir:
        return
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])[:n_samples]

    hazy_all, pred_all, gt_all = [], [], []
    for fname in files:
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_image_pair(os.path.join(hazy_dir, fname), gp)
        if hazy is None:
            continue
        pred = predict_full(model, hazy, tta=tta)
        hazy_all.append(hazy)
        pred_all.append(pred)
        gt_all.append(gt)

    if not hazy_all:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    ax1.hist(np.concatenate([h.flatten() for h in hazy_all]), bins=60,
             alpha=0.5, label='Hazy', color='orange', density=True)
    ax1.hist(np.concatenate([p.flatten() for p in pred_all]), bins=60,
             alpha=0.5, label='Predicted', color='blue', density=True)
    ax1.hist(np.concatenate([g.flatten() for g in gt_all]), bins=60,
             alpha=0.5, label='GT', color='green', density=True)
    ax1.set_title('Pixel Intensity Distribution', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(alpha=0.3)
    ax1.set_xlabel('Pixel Value')
    ax1.set_ylabel('Density')

    def sharp(imgs):
        scores = []
        for im in imgs[:20]:
            gray = cv2.cvtColor((im * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
            scores.append(cv2.Laplacian(gray, cv2.CV_64F).var())
        return np.mean(scores)

    vals = [sharp(hazy_all), sharp(pred_all), sharp(gt_all)]
    bars = ax2.bar(['Hazy', 'Predicted', 'GT'], vals,
                   color=['#FF9800', '#2196F3', '#4CAF50'], alpha=0.85,
                   edgecolor='black', linewidth=0.5)
    ax2.bar_label(bars, fmt='%.1f', fontsize=11)
    ax2.set_title('Sharpness (Laplacian Variance)', fontsize=13, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Pixel distribution saved: {save_path}")

# ═══════════════════════════════════════════════════════════════
# SECTION 12 — TRAINING HISTORY VISUALIZATION
# ═══════════════════════════════════════════════════════════════
def plot_training_history(csv_path, save_path):
    """Plot combined training curves from all stages."""
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        return

    if not csv_path or not os.path.exists(csv_path):
        print("  Training history CSV not found, skipping plot")
        return

    # Read CSV
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)

    if not rows:
        return

    stages = {}
    for row in rows:
        s = int(row['stage'])
        if s not in stages:
            stages[s] = {'epochs': [], 'loss': [], 'val_loss': [],
                         'psnr': [], 'val_psnr': [],
                         'ssim': [], 'val_ssim': [],
                         'mae': [], 'val_mae': []}
        ge = int(row.get('global_epoch', row['epoch']))
        stages[s]['epochs'].append(ge)
        stages[s]['loss'].append(float(row.get('loss', 0)))
        stages[s]['val_loss'].append(float(row.get('val_loss', 0)))
        stages[s]['psnr'].append(float(row.get('psnr_metric', 0)))
        stages[s]['val_psnr'].append(float(row.get('val_psnr_metric', 0)))
        stages[s]['ssim'].append(float(row.get('ssim_metric', 0)))
        stages[s]['val_ssim'].append(float(row.get('val_ssim_metric', 0)))
        stages[s]['mae'].append(float(row.get('mae_metric', 0)))
        stages[s]['val_mae'].append(float(row.get('val_mae_metric', 0)))

    colors = {1: ('#2196F3', '#0D47A1'), 2: ('#4CAF50', '#1B5E20'),
              3: ('#FF9800', '#E65100')}

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    metrics = [('loss', 'val_loss', 'Loss'),
               ('psnr', 'val_psnr', 'PSNR (dB)'),
               ('ssim', 'val_ssim', 'SSIM'),
               ('mae', 'val_mae', 'MAE')]

    for ax, (train_key, val_key, title) in zip(axes.flat, metrics):
        for s, data in sorted(stages.items()):
            c1, c2 = colors.get(s, ('#999', '#666'))
            ax.plot(data['epochs'], data[train_key], color=c1,
                    label=f'Train S{s}', linewidth=1.5)
            ax.plot(data['epochs'], data[val_key], color=c2,
                    linestyle='--', label=f'Val S{s}', linewidth=1.5)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(alpha=0.3)
        ax.set_xlabel('Global Epoch')

    plt.suptitle('ECA-LDNet — Full Training History (All Stages)',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Training curves saved: {save_path}")


In [ ]:

# ═══════════════════════════════════════════════════════════════
# SECTION 13 — COMPARISON TABLE (vs Other Models)
# ═══════════════════════════════════════════════════════════════
def generate_comparison_table(our_results, model, save_path):
    """Generate comparison table vs published dehazing methods on RESIDE benchmark."""

    # Published SOTS Indoor results from papers
    comparison_data = [
        # Method, Year, Attention, Loss, Indoor PSNR, Indoor SSIM, Params
        ('DCP (He et al.)', 2009, 'None', 'Prior-based', 16.62, 0.818, '—'),
        ('DehazeNet', 2016, 'None', 'MSE', 21.14, 0.847, '0.009M'),
        ('MSCNN', 2016, 'None', 'MSE', 17.57, 0.810, '0.008M'),
        ('AOD-Net', 2017, 'None', 'MSE', 19.06, 0.850, '0.002M'),
        ('GFN', 2018, 'None', 'L1+SSIM', 21.55, 0.844, '0.530M'),
        ('GridDehazeNet', 2019, 'Attention', 'L1+SSIM', 32.16, 0.984, '0.956M'),
        ('FFA-Net', 2020, 'CBAM (Channel+Pixel)', 'L1+SSIM+Perceptual', 36.39, 0.989, '4.456M'),
        ('DehazeFormer-S', 2023, 'Coordinate Attention', 'L1+SSIM+Perceptual', 30.29, 0.966, '1.283M'),
        ('DehazeFormer-B', 2023, 'Coordinate Attention', 'L1+SSIM+Perceptual', 33.26, 0.983, '25.44M'),
        ('DehazeFormer-L', 2023, 'Coordinate Attention', 'L1+SSIM+Perceptual', 34.77, 0.987, '25.44M'),
    ]

    # Get our results
    our_psnr_in = 0.0
    our_ssim_in = 0.0
    our_psnr_out = 0.0
    our_ssim_out = 0.0
    our_psnr_r6k = 0.0
    our_ssim_r6k = 0.0

    for name, records in our_results.items():
        if records:
            avg_p = np.mean([r['PSNR'] for r in records])
            avg_s = np.mean([r['SSIM'] for r in records])
            if 'Indoor' in name:
                our_psnr_in = avg_p
                our_ssim_in = avg_s
            elif 'Outdoor' in name:
                our_psnr_out = avg_p
                our_ssim_out = avg_s
            elif 'RESIDE' in name or 'R6K' in name:
                our_psnr_r6k = avg_p
                our_ssim_r6k = avg_s

    params_str = f'{model.count_params() / 1e6:.3f}M'
    our_psnr = our_psnr_in if our_psnr_in > 0 else our_psnr_r6k
    our_ssim = our_ssim_in if our_ssim_in > 0 else our_ssim_r6k

    comparison_data.append(
        ('ECA-LDNet (Ours)', 2025, 'ECA + PixelAttention',
         'Charbonnier+SSIM+Edge+FFT+VGG', round(our_psnr, 2), round(our_ssim, 4), params_str)
    )

    # Write CSV
    with open(save_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Method', 'Year', 'Attention_Type', 'Loss_Function',
                         'SOTS_Indoor_PSNR', 'SOTS_Indoor_SSIM', 'Params'])
        for row in comparison_data:
            writer.writerow(row)

    # Print table
    print(f"\n{'═' * 90}")
    print(f"  COMPARISON TABLE — RESIDE SOTS Indoor Benchmark")
    print(f"{'═' * 90}")
    print(f"  {'Method':<25} {'Year':<6} {'Attention':<25} {'PSNR':>8} {'SSIM':>8} {'Params':>10}")
    print(f"  {'─' * 85}")
    for row in comparison_data:
        marker = ' ★' if 'Ours' in row[0] else ''
        print(f"  {row[0]:<25} {row[1]:<6} {row[2]:<25} {row[4]:>8.2f} {row[5]:>8.4f} {row[6]:>10}{marker}")
    print(f"{'═' * 90}")

    # Ablation summary
    print(f"\n{'═' * 70}")
    print(f"  ABLATION STUDY SUMMARY — ECA-LDNet Components")
    print(f"{'═' * 70}")
    print(f"  {'Component':<35} {'Contribution':<35}")
    print(f"  {'─' * 65}")
    print(f"  {'ECA (Channel Attention)':<35} {'Channel-wise feature recalibration':<35}")
    print(f"  {'PixelAttention (Spatial)':<35} {'Spatial focus on hazy regions':<35}")
    print(f"  {'Physics Correction Layer':<35} {'ASM-guided refinement (J=I-A/t+A)':<35}")
    print(f"  {'Depthwise Separable Conv':<35} {'Param efficiency (1.48M total)':<35}")
    print(f"  {'VGG Perceptual Loss':<35} {'Feature-level quality matching':<35}")
    print(f"  {'Multi-Stage Training (3 stages)':<35} {'Progressive refinement S1→S2→S3':<35}")
    print(f"  {'Charbonnier + Edge + FFT Loss':<35} {'Robust pixel + frequency learning':<35}")
    print(f"{'═' * 70}")

    # Our detailed results
    print(f"\n{'═' * 70}")
    print(f"  ECA-LDNet RESULTS ON ALL BENCHMARKS")
    print(f"{'═' * 70}")
    if our_psnr_in > 0:
        print(f"  SOTS Indoor  : PSNR={our_psnr_in:.4f} dB  SSIM={our_ssim_in:.4f}")
    if our_psnr_out > 0:
        print(f"  SOTS Outdoor : PSNR={our_psnr_out:.4f} dB  SSIM={our_ssim_out:.4f}")
    if our_psnr_r6k > 0:
        print(f"  RESIDE-6K    : PSNR={our_psnr_r6k:.4f} dB  SSIM={our_ssim_r6k:.4f}")
    print(f"  Parameters   : {model.count_params():,} ({params_str})")
    print(f"{'═' * 70}")

    print(f"  Comparison table saved: {save_path}")
    return comparison_data

# ═══════════════════════════════════════════════════════════════
# SECTION 14 — FINAL REPORT GENERATION
# ═══════════════════════════════════════════════════════════════
def generate_final_report(our_results, model, latency, flops, save_path):
    """Generate comprehensive text report."""
    params = model.count_params()
    with open(save_path, 'w') as f:
        f.write("=" * 70 + "\n")
        f.write("ECA-LDNet — Comprehensive Testing Report\n")
        f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 70 + "\n\n")

        f.write("MODEL DETAILS\n")
        f.write(f"  Architecture  : ECA-LDNet (U-Net + Dual Attention + Physics)\n")
        f.write(f"  Parameters    : {params:,} ({params / 1e6:.3f}M)\n")
        f.write(f"  Input Size    : {IMG_SIZE}x{IMG_SIZE}x3\n")
        f.write(f"  Attention     : ECA (Channel) + PixelAttention (Spatial)\n")
        if flops:
            f.write(f"  FLOPs         : {flops:,} ({flops / 1e9:.2f}G)\n")
        if latency:
            f.write(f"  Latency       : {latency['avg_ms']:.1f} ± {latency['std_ms']:.1f} ms\n")
            f.write(f"  FPS           : {latency['fps']:.1f}\n")
        f.write("\n")

        f.write("LOSS FUNCTION\n")
        f.write("  Stage 1-2: 0.35×Charbonnier + 0.20×SSIM + 0.10×Edge + 0.10×FFT + 0.25×VGG\n")
        f.write("  Stage 3:   0.35×Charbonnier + 0.20×L1 + 0.20×SSIM + 0.15×MS-SSIM + 0.10×Edge\n\n")

        f.write("TRAINING DETAILS\n")
        f.write("  Stage 1: 60 epochs, LR=1e-3, AdamW, batch=16\n")
        f.write("  Stage 2: 30 epochs, LR=1e-4, AdamW, batch=16\n")
        f.write("  Stage 3: 40 epochs, LR=5e-5, AdamW, batch=16\n")
        f.write("  Data: RESIDE-6K (6K) + ITS (13,990) = 19,990 pairs\n\n")

        f.write("BENCHMARK RESULTS\n")
        f.write("-" * 60 + "\n")
        for name, records in our_results.items():
            if records:
                psnrs = [r['PSNR'] for r in records]
                ssims = [r['SSIM'] for r in records]
                maes = [r['MAE'] for r in records]
                f.write(f"  {name} ({len(records)} images):\n")
                f.write(f"    PSNR  : {np.mean(psnrs):.4f} ± {np.std(psnrs):.4f} dB\n")
                f.write(f"    SSIM  : {np.mean(ssims):.4f} ± {np.std(ssims):.4f}\n")
                f.write(f"    MAE   : {np.mean(maes):.6f}\n\n")
        f.write("=" * 70 + "\n")
    print(f"  Final report saved: {save_path}")




In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 15 — MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════

def main():
    args = parse_args()

    # Output directory
    out_dir = args.output
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir, 'csv'), exist_ok=True)
    os.makedirs(os.path.join(out_dir, 'plots'), exist_ok=True)

    # Detect model path
    model_path = args.model
    if not model_path:
        # Try common locations
        candidates = [
            '/kaggle/input/models/codeninjalucky/stage-5-model/tensorflow2/default/1/stage5_final_ema.keras',
            './models/stage3_best.keras',
            '/kaggle/working/models/stage3_best.keras',
            '/kaggle/working/stage3_best.keras',
            '../stage_3_version_1/stage3_best.keras',
        ]
        for c in candidates:
            if os.path.exists(c):
                model_path = c
                break

    if not model_path or not os.path.exists(model_path):
        print(f"\n  ERROR: Model file not found!")
        print(f"  Tried: {model_path}")
        print(f"  Use --model path/to/stage3_best.keras")
        sys.exit(1)

    # Load model
    print(f"\nLoading model: {model_path}")
    model = tf.keras.models.load_model(
        model_path, custom_objects=CUSTOM_OBJECTS, compile=False)
    total_params = model.count_params()
    print(f"Model loaded: {total_params:,} params ({total_params / 1e6:.3f}M)")

    # Setup datasets
    paths = setup_dataset_paths(args)

    # ═══════════════════════════════════════════════════════════
    # TEST 1: FLOPs & Latency
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'═' * 60}")
    print("TEST 1 — FLOPs & Latency Analysis")
    print(f"{'═' * 60}")
    flops = compute_model_flops(model)
    if flops:
        print(f"  FLOPs: {flops:,} ({flops / 1e9:.2f} GFLOPs)")
    latency = measure_latency(model, n_runs=30)
    print(f"  Latency: {latency['avg_ms']:.1f} ± {latency['std_ms']:.1f} ms")
    print(f"  FPS: {latency['fps']:.1f}")

    # ═══════════════════════════════════════════════════════════
    # TEST 2-4: Full Dataset Evaluations
    # ═══════════════════════════════════════════════════════════
    tta = not args.no_tta
    all_results = {}

    # RESIDE-6K Test
    r6k = evaluate_dataset(
        model, paths['R6K_TEST_HAZY'], paths['R6K_TEST_GT'],
        'RESIDE-6K', limit=args.limit, tta=tta,
        save_csv=os.path.join(out_dir, 'csv', 'results_reside6k.csv'))
    if r6k:
        all_results['RESIDE-6K'] = r6k

    # SOTS Indoor
    sots_in = evaluate_dataset(
        model, paths['SOTS_IN_HAZY'], paths['SOTS_IN_GT'],
        'SOTS Indoor', limit=args.limit, tta=tta,
        save_csv=os.path.join(out_dir, 'csv', 'results_sots_indoor.csv'))
    if sots_in:
        all_results['SOTS Indoor'] = sots_in

    # SOTS Outdoor
    sots_out = evaluate_dataset(
        model, paths['SOTS_OUT_HAZY'], paths['SOTS_OUT_GT'],
        'SOTS Outdoor', limit=args.limit, tta=tta,
        save_csv=os.path.join(out_dir, 'csv', 'results_sots_outdoor.csv'))
    if sots_out:
        all_results['SOTS Outdoor'] = sots_out

    # ═══════════════════════════════════════════════════════════
    # TEST 5: TTA vs No-TTA Comparison
    # ═══════════════════════════════════════════════════════════
    compare_tta(model, paths['R6K_TEST_HAZY'], paths['R6K_TEST_GT'],
                'RESIDE-6K', n_samples=50,
                save_csv=os.path.join(out_dir, 'csv', 'tta_comparison.csv'))

    # ═══════════════════════════════════════════════════════════
    # TEST 6-8: Visual Comparisons
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'═' * 60}")
    print("GENERATING VISUAL COMPARISONS...")
    print(f"{'═' * 60}")

    plot_visual_comparison(
        model, paths['R6K_TEST_HAZY'], paths['R6K_TEST_GT'],
        os.path.join(out_dir, 'plots', 'visual_reside6k.png'),
        'ECA-LDNet — RESIDE-6K Results', tta=tta)

    plot_visual_comparison(
        model, paths['SOTS_IN_HAZY'], paths['SOTS_IN_GT'],
        os.path.join(out_dir, 'plots', 'visual_sots_indoor.png'),
        'ECA-LDNet — SOTS Indoor Results', tta=tta)

    plot_visual_comparison(
        model, paths['SOTS_OUT_HAZY'], paths['SOTS_OUT_GT'],
        os.path.join(out_dir, 'plots', 'visual_sots_outdoor.png'),
        'ECA-LDNet — SOTS Outdoor Results', tta=tta)

    # ═══════════════════════════════════════════════════════════
    # TEST 9: Error Heatmaps
    # ═══════════════════════════════════════════════════════════
    plot_error_heatmaps(
        model, paths['R6K_TEST_HAZY'], paths['R6K_TEST_GT'],
        os.path.join(out_dir, 'plots', 'error_heatmap_r6k.png'),
        n_samples=50, tta=tta)

    # ═══════════════════════════════════════════════════════════
    # TEST 10: Pixel Distribution & Sharpness
    # ═══════════════════════════════════════════════════════════
    plot_pixel_distribution(
        model, paths['R6K_TEST_HAZY'], paths['R6K_TEST_GT'],
        os.path.join(out_dir, 'plots', 'pixel_distribution.png'),
        n_samples=50)

    # ═══════════════════════════════════════════════════════════
    # TEST 11: Training History Plot
    # ═══════════════════════════════════════════════════════════
    history_csv = args.history_csv
    if not history_csv:
        candidates = [
            '/kaggle/input/models/codeninjalucky/models-dataset/tensorflow2/default/1/combined_training_history.csv',
            '../final_model/combined_training_history.csv',
            '/kaggle/working/combined_training_history.csv',
        ]
        for c in candidates:
            if os.path.exists(c):
                history_csv = c
                break
    plot_training_history(
        history_csv,
        os.path.join(out_dir, 'plots', 'training_curves.png'))

    # ═══════════════════════════════════════════════════════════
    # TEST 12: Comparison Table & Ablation
    # ═══════════════════════════════════════════════════════════
    generate_comparison_table(
        all_results, model,
        os.path.join(out_dir, 'csv', 'comparison_table.csv'))

    # ═══════════════════════════════════════════════════════════
    # TEST 13: Final Report
    # ═══════════════════════════════════════════════════════════
    generate_final_report(
        all_results, model, latency, flops,
        os.path.join(out_dir, 'final_report.txt'))

    # ═══════════════════════════════════════════════════════════
    # FINAL SUMMARY
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'═' * 70}")
    print(f"  ALL TESTS COMPLETE — ECA-LDNet")
    print(f"{'═' * 70}")
    print(f"  Model     : {total_params:,} params ({total_params / 1e6:.3f}M)")
    print(f"  Latency   : {latency['avg_ms']:.1f} ms ({latency['fps']:.1f} FPS)")
    for name, records in all_results.items():
        if records:
            avg_p = np.mean([r['PSNR'] for r in records])
            avg_s = np.mean([r['SSIM'] for r in records])
            print(f"  {name:<15}: PSNR={avg_p:.4f} dB  SSIM={avg_s:.4f}")
    print(f"\n  Results saved to: {out_dir}/")
    print(f"{'═' * 70}")
    print("  TESTING COMPLETE ✓")


In [ ]:
main()